# Poromechanics
This tutorial illustates how to use PorePy's poromechanical model to simulate frictional fracture deformation during fluid injection. For more information on the mathematical model and the simulation setup, we refer to the PorePy tutorial on
<a href="https://github.com/pmgbergen/porepy/blob/develop/tutorials/poromechanics.ipynb" target="_blank">poromechanics</a>.

This tutorial covers:
1. Seting up and running a 3d simulation with poromechanics and frictional fracture deformation
2. Defining fractures with an eliptic shape in 3d
3. Use relatively simple functionality to mimic fluid injection through a well
4. Use line search to stabilize the simulations - this can be critical for the numerical stability of the simulation


## Import of helper mixins
PorePy has several helper mixins that provide convenience methods for setting up simulations. Many of these are located in the folder 
<a href="https://github.com/pmgbergen/porepy/tree/develop/src/porepy/applications" target="_blank">applications</a>. In this case, we import mixin classes for setting boundary and initial conditions.

PorePy also provides a set of full examples, located in the folder <a href="https://github.com/pmgbergen/porepy/tree/develop/src/porepy/examples" target="_blank">examples</a>. These examples aimed to illustrate how PorePy can be adapted to set up various simulations, and if possible, it is recommended to use an existing example as the starting point for setting up a new simulation. The below setup was modified from an example that prescribe thermo-hydromechanical deformation in a 
<a href="https://github.com/pmgbergen/porepy/tree/develop/src/porepy/examples/geothermal_reservoir.py" target="_blank">geothermal reservoir</a>, and we will borrow some mixin classes from that setup.

In [ ]:
# The usual imports.
import numpy as np
import porepy as pp

# Ready setups for boundary and initial conditions.
from porepy.applications.boundary_conditions.model_boundary_conditions import (
    HydrostaticBoundaryPressureValues,
    BoundaryConditionsMechanicsNeumann,
    LithostaticBoundaryStressValues,
)
from porepy.applications.initial_conditions.model_initial_conditions import (
    InitialConditionHydrostaticPressureValues,
)
# Mixins for well boundary conditions - see more information below.
from porepy.examples.geothermal_reservoir import (
    WellBoundaryConditions,
    NeumannWellBCsFirstTimeInterval,
) 
# A mixin for storing the fracture deformation history - see below for example usage.
from porepy.viz.data_saving_model_mixin import FractureDeformationExporting

# Finally, import an line search method that will be used to solve the nonlinear problem.
# For more information on this method, see the PorePy tutorial on solution strategies.
from porepy.numerics.nonlinear import line_search

## Set up a logger
For long-running simulations, or simulations that involve strong non-linearities like fracture deformation, it is often convenient to have the simulator print progress reports. PorePy has some support for this using Python's standard logging functionality. The logging framework in PorePy is very far from being complete or self-consistent, but adding the following lines to a runscript will often pay off.

In [ ]:

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

## Set the domain and fracture geometry
As for the tutorial on [flow and transport](../flow_and_transport/flow_and_transport_2d.ipynb), the domain and fracture geometry are set by a mixin class. In this case, we set the sizes directly in the overridden models. 

For 3d geometries, two types of fractures are available: 
* Elliptic fractures are specified in terms of their center, major and minor axes, and a set of rotation operations (see the class documentation for more information).
* Polygonal fractures are defined in terms of their vertices. The polygons should form planar objects and be convex (non-convex fractures might work, but proceed with caution if you try this).

In [ ]:
# Parametrized value used to define the problem geometry.
DOMAIN_SIZE = 1000

class Geometry:
    # Mixin class to set the geometry of the problem.
    def set_domain(self):

        self._domain = pp.Domain(
            {
                "xmin": 0,
                "xmax": DOMAIN_SIZE,
                "ymin": 0,
                "ymax": DOMAIN_SIZE,
                "zmin": 0,
                "zmax": DOMAIN_SIZE,
            }
        )

    def set_fractures(self):
        f_1 = pp.EllipticFracture(
            DOMAIN_SIZE * np.array([0.5, 0.5, 0.5]),  # Center of the fracture
            DOMAIN_SIZE * 0.4,  # Semi-major axis
            DOMAIN_SIZE * 0.3,  # Semi-minor axis
            0,  # Angle of the semi-major axis with respect to the x-axis
            0,  # Strike angle measured from the y-axis in the xy-plane
            0,  # Dip angle
        )
        f_2 = pp.EllipticFracture(
            DOMAIN_SIZE * np.array([0.7, 0.5, 0.5]),
            DOMAIN_SIZE * 0.3,
            DOMAIN_SIZE * 0.2,
            0,
            0,
            np.radians(90),  # NB: All angles in radians
        )
        fractures = [f_1, f_2]
        self._fractures = fractures


## Well geometries
Though PorePy lacks functionality to handle most complexities related to flow in wells, there is a simple model that allows for the geometric representation of wells, together with fluid flow in the wells. This comes with several assumptions, notably, fluid can only exit the well where it intersects with a fracture; the quality of this assumption may depend on the quality of the host rock.

Wells are specified by overriding the method set_well_network, as shown below. The well trajectory is specfied as a 3 x n array of coordinates, where the first coordinate (column) should touch a domain boundary, the last defines the bottom of the well, and the intermediate column can be used to specify a trajectory which is not a straight line. It should be noted that the current treatment of polyline wells is known to be faulty and should be used with caution (fixes should be available in the next release).

In [ ]:
class WellGeometry:
    def set_well_network(self):
        """Set the well geometry"""

        well_1 = pp.Well(
            np.array(
                [
                    [0.3 * DOMAIN_SIZE, 0.3 * DOMAIN_SIZE],
                    [0.5 * DOMAIN_SIZE, 0.5 * DOMAIN_SIZE],
                    [DOMAIN_SIZE, 0.2 * DOMAIN_SIZE],
                ]
            ),
            tags={"well_name": "injection_well"},
        )
        self._wells = [well_1]

        mesh_size = self.params.get("well_mesh_size", {"mesh_size": 0.1 * DOMAIN_SIZE})
        self.well_network = pp.WellNetwork3d(
            domain=self._domain, wells=self._wells, parameters=mesh_size
        )

After generating the mesh (which does not happen before the start of the simulation below), and after some manipulation in Paraview, the problem geometry and mesh can be visualized as follows:

<img src='mesh.png' width=900>

The figure shows the injection well (orange), the fractures (brown), and the domain boundary (black lines). In the fractures and in part of the domain, the computational mesh is also shown. Please note that, to limit the computational effort needed to run the simulations, this mesh is relatively coarse - for practical simulations, a finer mesh will often be preferrable.

## Boundary conditions
Specifying boundary conditions for coupled simulations is a complex task. Here we use 

In [ ]:
class BoundaryConditions(
    WellBoundaryConditions,
    NeumannWellBCsFirstTimeInterval,
    # Hydrostatic conditions for the pressure on the domain boundaries.
    HydrostaticBoundaryPressureValues,
    # Impose Neumann type conditions on almost the entire boundary for the mechanics.
    # See the documentation of this class for details, including how rigid body motion
    # is prevented.
    BoundaryConditionsMechanicsNeumann,
    # Set lithostatic conditions for the stress on the domain boundaries through
    # multipliers, see the parameter 'lithostatic_stress_multiplier' in the parameters
    # of the model.
    LithostaticBoundaryStressValues,
):
    pass

## Simulation class
Now we can gather all mixins into a full simulation class. We will augment the standard visualization workflow to store data on the fracture deformation state, which can be used for visualization level, see comments below.

**Important**: The methods overridden below are called as part of the workflow in a PorePy simulation. There are several of these methods, mainly located in the class
<a href="https://github.com/pmgbergen/porepy/tree/develop/src/porepy/models/solution_strategy.py" target="_blank">SolutionStrategy</a>. The methods are designed to be overridden so as to adapt the behavior of a simulation, and most of the multiphysics base models provide such adaptations. In most cases it is critical that, when overriding such methods, a call to the same method in the super class (see below for examples) is included. This is the mechanism that makes our adaptation an addition, and not a replacement, of the behavior of the existing model. Failure to include the super call can lead to unexpected behavior including simulation errors and the simulation effectively solving a different model than what is expected. There are of course cases where the correction option is to replace the default behavior, but this should be done with care.


Points to note:
* How to augment the data saving methods that are used for export and visualization during a simulation.
* What to do, and not to do, when overriding methods that form part of the standard simulation workflow.

In [ ]:
class SimulationSetup(
    # Mixins for the geometry and the well trajectory
    Geometry,
    WellGeometry,
    # Activate gravity.
    pp.constitutive_laws.GravityForce,
    # Set the fracture permeability through a cubic law. The matrix permeability will
    # be set through the solid constants specified below.
    pp.constitutive_laws.CubicLawPermeability,
    # Mixin for the boundary conditions, see above for details.
    BoundaryConditions,
    # Use hydrostatic conditions for the initial pressure. For the mechanics state we
    # use the default initial condition (zero displacements), and instead impose
    # conditions through an initialization phase described below.
    InitialConditionHydrostaticPressureValues,
    # This mixin introduces contact indicators that are used by the line search
    # algorithm specified in the nonlinear solver section below.
    pp.models.solution_strategy.ContactIndicators,
    # A helper mixin for postprocessing information on fracture deformation. We use the
    # result of this in the method after_nonlinear convergence, see below.
    FractureDeformationExporting,
    # Finally the base model. All methods not overriden by the above mixins will be
    # taken from this class.
    pp.Poromechanics,
):
    def initialize_data_saving(self):
        # First, call the super method to ensure that whatever data saving is
        # implemented in the parent classes is properly initialized. As a rule of thumb
        # (that should occasionally be broken, but is a good starting point), such super
        # calls should be made when overriding such methods. Most often, the order of
        # the super calls (including whether to put the super call at the beginning or
        # the end of the method) is not important (this will be the case when the
        # operations at different levels are independent). In some cases the order can
        # be critical - if so, careful checking of the code and its behavior is called
        # for. In this case, it does not matter what we do.
        super().initialize_data_saving()

        # Create attributes to store the history of the displacement jumps accross the
        # fractures, as well as the slip tendency (which is a measure of how close the
        # fracture is to slipping). These attributes will be filled in the method
        # after_nonlinear_convergence.
        self.jump_history = {sd: [] for sd in self.mdg.subdomains(dim=self.nd - 1)}
        self.jump_history_time = []
        self.slip_tendency = {sd: [] for sd in self.mdg.subdomains(dim=self.nd - 1)}

    def after_nonlinear_convergence(self):
        # Let the parent classes do whatever they need to do after nonlinear
        # convergence.
        super().after_nonlinear_convergence()

        # Call the method data_to_export, which was introduced by the mixin
        # FractureDeformationExporting, to get the data that is being saved at the
        # current time step. This will include the displacement jump across the
        # fractures, as well as the slip tendency. Store the values in the attributes
        # created above.
        data = self.data_to_export()
        values = {sd: {} for sd in self.mdg.subdomains(dim=self.nd - 1)}
        self.jump_history_time.append(self.time_manager.time)

        for sd, variable, value in data:
            if sd in values and variable == "displacement_jump":
                self.jump_history[sd].append(value)
            elif variable == "slip_tendency":
                self.slip_tendency[sd].append(value)

## Initialization of poromechanical models with fractures


In [37]:
# Define time schedule for the simulation.
schedule = np.array([0, pp.HOUR, 10 * pp.HOUR])

# Add initialization time interval.
dt_init = pp.YEAR
schedule += dt_init * 2.5
schedule = np.insert(schedule, 0, 0.0)

# Define injection pressures as list of len = schedule.size. For other protocol
# values, broadcasting of single values is used for simplicity. The following
# schedule is somewhat arbitrary, but meant to represent a ramping up of injection
# pressures over time. The initial low pressure represents a start from near
# hydrostatic conditions.

# We ramp up from 1e5 to 5e6 Pa during initialization (well is closed using a
# Neumann BC), then ramp up to 9e6 Pa at injection start (1 hour), then increase to
# 11e6 Pa after 10 hours, and finally to 15e6 Pa after 200 days.
injection_pressures = [1e5, 5e6, 9e6, 11e6]  # [Pa]
# injection_pressures = [1e5, 5e5, 5e5, 5e5, 5e5]  # --- IGNORE ---
# Convenient shortening of simulation schedule for quick simulations. The point is
# that injection_pressures must match the size of schedule.
schedule_length = schedule.size
schedule = schedule[:schedule_length]
injection_pressures = injection_pressures[:schedule_length]

time_manager = pp.TimeManager(
    schedule=schedule,
    dt_init=dt_init,
    constant_dt=False,
    dt_min_max=(0.1 * pp.MINUTE, max(pp.HOUR, dt_init)),
    iter_optimal_range=(6, 10),  # Allow more iterations than default.
    iter_relax_factors=(0.5, 1.8),  # More aggressive relaxation
)

In [39]:
solid_values = pp.solid_values.basalt
solid_values.update(
    {
        "dilation_angle": 0.1,  # [rad]
        # Uncomment next two lines to include elastic fracture deformation, aka
        # "Barton-Bandis" model for normal fracture deformation.
        # "fracture_normal_stiffness": 1.1e8,  # [Pa m^-1]
        # "maximum_elastic_fracture_opening": 1e-3,  # [m]
        "normal_permeability": 1.0e-10,  # [m^2]
        "residual_aperture": 1e-3,  # [m]
        "well_radius": 0.1,  # [m]
    }
)

In [40]:
model_params = {
    # Set time manager.
    "time_manager": time_manager,
    # Set physical parameters.
    "lithostatic_stress_multipliers": np.array([0.8, 1.2, 1.0]),
    "injection_well_pressures": injection_pressures,
    "production_well_pressures": pp.ATMOSPHERIC_PRESSURE,  # = 1.01325e5 Pa
    "material_constants": {
        "solid": pp.SolidConstants(**solid_values),  # type: ignore[arg-type]
        "fluid": pp.FluidComponent(**pp.fluid_values.water),  # type: ignore[arg-type]
        "numerical": pp.NumericalConstants(characteristic_displacement=1e-2),
    },
    "reference_variable_values": pp.ReferenceVariableValues(pressure=1e6),
    "units": pp.Units(m=1.0, kg=1.0e5, K=1.0),
    # Set geometry and meshing related parameters.
    "grid_type": "simplex",
    "meshing_arguments": {
        "cell_size": 0.7 * DOMAIN_SIZE,  # Base cell size for meshing.
        "cell_size_fracture": 0.3 * DOMAIN_SIZE,
        "cell_size_min": 0.1 * DOMAIN_SIZE,
    },
    "domain_sizes": 1.0,
    # Line search: Scale the indicator used for the local_line_search (see below)
    # adaptively to increase robustness.
    "adaptive_indicator_scaling": 1,
}

In [41]:
solver_params = {
    "prepare_simulation": True,
    "nl_max_iterations": 25,  # Max iterations of a nonlinear solver (Newton)
    "nl_convergence_inc_atol": 1e-7,  # Increment norm
    "nl_convergence_res_atol": 1e-7,  # Residual norm
    "nl_divergence_inc_atol": 1e12,
    "nl_divergence_res_atol": 1e12,
    # Line search / Solution Strategies. These are considered "advanced" options,
    # improving the robustness of the nonlinear solver at the cost of some
    # additional computational overhead. Delete/comment the following lines for the
    # default Newton's method.
    "nonlinear_solver": line_search.ConstraintLineSearchNonlinearSolver,
    # Set to 1 to use turn on a residual-based line search. This involves some extra
    # residual evaluations and may be quite costly.
    "global_line_search": 0,
    # Set to 0 to use turn off the tailored line search, see the class
    # ConstraintLineSearchNonlinearSolver. This line search is cheap and has proven
    # effective for (some versions of) this particular simulation setup.
    "local_line_search": 1,
}

In [42]:
model = SimulationSetup(model_params)
pp.run_time_dependent_model(model, solver_params)

INFO:porepy.fracs.simplex:Grid creation completed. Elapsed time 0.0234527587890625
INFO:porepy.fracs.simplex:Created 1 3-d grids with 2107 cells
INFO:porepy.fracs.simplex:Created 2 2-d grids with 122 cells
INFO:porepy.fracs.simplex:Created 1 1-d grids with 4 cells


INFO:porepy.numerics.fv.biot:Done with subproblem 0. Elapsed time 6.936313629150391
INFO:porepy.models.solution_strategy:Discretized in 7.906240701675415 seconds
INFO:porepy.models.run_models:
Time step 1 at time 3.2e+07 of 7.9e+07 with time step 3.2e+07
INFO:porepy.models.solution_strategy:Solved linear system in 1.14e+00 seconds.
INFO:porepy.numerics.nonlinear.nonlinear_solvers:Newton iteration number 1 of 25
INFO:porepy.numerics.nonlinear.nonlinear_solvers:Nonlinear increment norm: 2.38e+02, Nonlinear residual norm: 4.64e-02
INFO:porepy.models.solution_strategy:Solved linear system in 1.07e+00 seconds.
INFO:porepy.numerics.nonlinear.nonlinear_solvers:Newton iteration number 2 of 25
INFO:porepy.numerics.nonlinear.nonlinear_solvers:Nonlinear increment norm: 1.23e-02, Nonlinear residual norm: 9.34e-08
INFO:porepy.models.solution_strategy:Solved linear system in 1.05e+00 seconds.
INFO:porepy.numerics.nonlinear.nonlinear_solvers:Newton iteration number 3 of 25
INFO:porepy.numerics.nonlin

In [43]:
for sd, jump_history in model.jump_history.items():
    print(f"Fracture {sd.frac_num + 1}")
    for i, val in enumerate(jump_history):
        print(
            f"Time {model.jump_history_time[i]}: Max displacement jump = {model.units.convert_units(val.max(), 'm', to_si=True)} m"
        )
    for i, val in enumerate(model.slip_tendency[sd]):
        print(f"Time {model.jump_history_time[i]}: Slip tendency = {val.max()}")

Fracture 1
Time 31536000.0: Max displacement jump = 6.938893903907228e-18 m
Time 63072000.0: Max displacement jump = 6.938893903907228e-18 m
Time 78840000.0: Max displacement jump = 6.938893903907228e-18 m
Time 78843600.0: Max displacement jump = 0.0 m
Time 78847200.0: Max displacement jump = 0.0 m
Time 78850800.0: Max displacement jump = 0.0 m
Time 78857280.0: Max displacement jump = 0.0 m
Time 78868944.0: Max displacement jump = 5.421010862427522e-20 m
Time 78876000.0: Max displacement jump = 5.421010862427522e-20 m
Time 31536000.0: Slip tendency = 0.05847263618847161
Time 63072000.0: Slip tendency = 0.05571388096348523
Time 78840000.0: Slip tendency = 0.05502639190935546
Time 78843600.0: Slip tendency = 0.13911857107278125
Time 78847200.0: Slip tendency = 0.16120540687774423
Time 78850800.0: Slip tendency = 0.1838238157478089
Time 78857280.0: Slip tendency = 0.23473776423276138
Time 78868944.0: Slip tendency = 0.3442401229853842
Time 78876000.0: Slip tendency = 0.45776459725861557
F

In [228]:
model.mdg.subdomains()[2].nodes

array([[700.        , 700.        , 700.        , 700.        ,
        700.        , 700.        , 700.        , 700.        ,
        700.        , 700.        , 700.        , 700.        ,
        700.        , 700.        , 700.        , 700.        ,
        700.        , 700.        , 700.        , 700.        ,
        700.        , 700.        , 700.        , 700.        ],
       [300.        , 300.        , 500.        , 700.        ,
        700.        , 310.64099663, 360.75600831, 398.35124706,
        398.35124706, 500.        , 500.        , 601.64875294,
        601.64875294, 639.24886786, 689.36033407, 689.08606361,
        637.37231403, 500.00956876, 362.63246565, 310.91408529,
        442.10027196, 568.00075455, 557.70555623, 432.51105408],
       [500.        , 500.        , 200.        , 500.        ,
        500.        , 403.44883403, 284.65144707, 500.        ,
        500.        , 500.        , 500.        , 500.        ,
        500.        , 284.65854139, 40

In [188]:
f_2 = pp.EllipticFracture(
    DOMAIN_SIZE * np.array([0.7, 0.5, 0.5]),
    DOMAIN_SIZE * 0.3,
    DOMAIN_SIZE * 0.2,
    0,
    0,
    90,
)

In [193]:
f_2.dip_angle

90